# GPU tier smoke testRun this on Kaggle with a **GPU accelerator and Internet enabled** beforecommitting GPU hours to the twelve outstanding models.The point is to find out what breaks. Nothing here is a benchmark run: oneticker, ten sessions, one epoch. Every cell is independently runnable — eachcarries its own imports, its own data, and its own installs — so a cell thatfails can be re-run alone after a fix without replaying the notebook.Each cell ends in a single `PASS` or `FAIL` line. Cell 3 is expected to be thenoisy one; it reports and does not debug.**Ordering is deliberate.** Kronos runs first because it is the riskiest: aseparate repository rather than a pip package, a separate API, its owntokeniser, and OHLCV input where everything else takes a univariate series. Ifit fails, that is worth knowing before any budget is spent.Reference: `configs/models.yaml` and `src/sp500bench/models/registry.py` are thesource of truth for checkpoints. The five foundation entries are Chronos-2,TimesFM-2.5, Moirai-2.0, Kronos-small and TTM.

## What the smoke test has establishedKaggle, Tesla T4 15.6 GB, torch 2.10.0+cu128, CUDA 12.8. Three runs so far.| model | status | detail ||---|---|---|| **Kronos-small** | **PASS** | loads and runs || **Moirai-2.0** | **LOADED** | predictor constructs || **TimesFM-2.5** | **RESOLVED** | entry point is `timesfm.TimesFM3Forecaster` || Chronos-2 | **UNTESTED** | every failure so far had an environment cause || TTM | **UNTESTED** | same || DLinear (cell 4) | fixed, not yet re-run | wrong argument name, corrected |### The root cause: the installs were moving torchCell 3's installs dragged torch from **2.10.0+cu128 to 2.14.0+cu130**. Kaggle's`torchvision 0.25.0` and `torchaudio 2.10.0` are compiled against 2.10.0, sothey end up partially initialised against the new binary. `transformers` importstorch on the way in and breaks with it. That is how Chronos-2 and TTM came toreport```ImportError: cannot import name 'PreTrainedModel' from 'transformers'```under transformers **4.56.0** — a version where `PreTrainedModel` certainlyexists. The error names transformers; the fault was the torch stack underneathit. The earlier `_torch_version` and `is_tf_available` failures are the samestory told through different symbols.**Fix: a pip constraints file.** Constraints pin a version *without adding arequirement*, which is exactly right here — nothing being installed wants torch,and nothing may move it:```!printf "torch==2.10.0\ntorchvision==0.25.0\ntorchaudio==2.10.0\n" > /tmp/c.txt!pip install -q -c /tmp/c.txt chronos-forecasting timesfm uni2ts granite-tsfm!pip install -q -c /tmp/c.txt "transformers<5"```Cell 3 prints torch, torchvision, torchaudio and transformers **before andafter** the install and marks anything that moved, so a regression is visible onthe spot rather than three symptoms downstream. If the stack moves anyway thecell stops instead of reporting model results from a broken environment.**If a package will not install under the constraint, that is the answer.** Thecell names which one and stops. A package that cannot coexist with Kaggle'storch 2.10 stack is a genuine incompatibility and a go/no-go input — either itsupports the platform or it needs its own session with its own image. It is notsomething to route around by letting torch move.### Restarting the kernel is not enough — stop the session**A kernel restart does not reset pip state on Kaggle.** It clears Python stateand leaves `site-packages` exactly as the previous cell's pip left it. After anunconstrained install, restarting gives a fresh interpreter on top of a brokenenvironment: the symptoms move and the cause does not.**Use the power button and stop the session entirely** to get the clean imageback. Then run cell 1 and the cell you are testing — nothing else. Cell 2imports transformers, and pip cannot replace a module that is already imported.This is also why the second run's transformers pin was never exercised: pip put4.56.0 on disk while the kernel kept serving 5.0.0.### Cell 4 — wrong argument, now corrected```TypeError: Trainer.__init__() got an unexpected keyword argument'num_workers_loader'````num_workers_loader` does not exist in neuralforecast 3.2.1. Anything the modelconstructor does not name falls into `**trainer_kwargs` and is passed to`pl.Trainer`, which rejects it. Verified against the 3.2.1 source rather thanassumed: `grep -rn num_workers_loader neuralforecast/` returns nothing, and`DLinear.__init__` names `dataloader_kwargs=None`. The correct call is`DLinear(dataloader_kwargs={"num_workers": 0}, accelerator="gpu", devices=1)` —dataloader settings in the dict, trainer settings loose.Zero workers remains a workaround for cell ordering, not a fix. **The realneural-tier run belongs in its own session.**### Two things Kronos passing has already settledBoth are already in `registry.py` and `configs/models.yaml`; the smoke testconfirms them against the running model.- It emits **price-level OHLCV**, so returns must be derived from the predicted  close rather than read off directly.- It samples autoregressively, so its **CRPS must come from sampled paths**, not  a closed-form quantile head. `sample_count=5` here is a smoke test; a real run  needs far more, and that is a compute cost the budget has to carry.### Suggested next runStop the session between each of these, not just restart the kernel.1. Cell 1, then **cell 3 alone** — the constrained install. Settles Chronos-2   and TTM, and is the only step that can produce new information.2. Cell 1, then cell 2 — reconfirms Kronos.3. Cell 1, then cell 4 — confirms the nine quantile columns.

In [ ]:
# ============================================================# CELL 1 - ENVIRONMENT# ============================================================!pip install -q neuralforecast statsforecast pyyamlimport sysprint("=" * 60)ok = Truetry:    import torch    print(f"torch              {torch.__version__}")    cuda = torch.cuda.is_available()    print(f"CUDA available     {cuda}")    if cuda:        print(f"GPU                {torch.cuda.get_device_name(0)}")        print(f"CUDA version       {torch.version.cuda}")        total = torch.cuda.get_device_properties(0).total_memory / 1e9        print(f"GPU memory         {total:.1f} GB")    else:        print("GPU                none - accelerator is off or unavailable")        ok = Falseexcept Exception as exc:    print(f"torch import failed: {type(exc).__name__}: {exc}")    ok = Falsefor name in ("neuralforecast", "statsforecast", "yaml"):    try:        module = __import__(name)        print(f"{name:<19}{getattr(module, '__version__', 'ok')}")    except Exception as exc:        print(f"{name:<19}FAILED: {type(exc).__name__}: {exc}")        ok = Falseprint(f"python             {sys.version.split()[0]}")print("=" * 60)print("PASS - environment ready" if ok else      "FAIL - no GPU or a core package is missing; fix before running cells 2-4")

In [ ]:
# ============================================================# CELL 2 - KRONOS  (riskiest first: separate repo, separate API)# ============================================================# This cell imports transformers. If you intend to run cell 3 afterwards, stop# the session first (power button) - a kernel restart clears Python state but# leaves pip's site-packages as it was, so cell 3's constraints cannot take# effect on top of it.# ============================================================import sysimport warningsimport numpy as npimport pandas as pdwarnings.filterwarnings("ignore")TICKER = "AAPL"CONTEXT = 400          # Kronos-small documents a 512-session context limitPRED_LEN = 10SAMPLE_COUNT = 5# Idempotent: the guard is shell-side so the cell can be re-run alone# without git erroring on an existing directory.![ -d Kronos ] || git clone --depth 1 https://github.com/shiyu-coder/Kronos.gitif "Kronos" not in sys.path:    sys.path.append("Kronos")def load_bars(ticker, n):    """OHLCV bars. Kronos is the one model here that takes them natively."""    try:        import yfinance as yf        frame = yf.download(ticker, period="3y", progress=False, auto_adjust=True)        if len(frame):            if isinstance(frame.columns, pd.MultiIndex):                frame.columns = frame.columns.get_level_values(0)            frame = frame.rename(columns=str.lower)[                ["open", "high", "low", "close", "volume"]].dropna()            return frame.tail(n), "yfinance"    except Exception as exc:        print(f"  yfinance unavailable ({type(exc).__name__}); using synthetic bars")    rng = np.random.default_rng(20260902)    close = 100 * np.exp(np.cumsum(rng.normal(0, 0.015, n)))    frame = pd.DataFrame({        "open": close * (1 + rng.normal(0, 0.002, n)),        "high": close * (1 + abs(rng.normal(0, 0.006, n))),        "low": close * (1 - abs(rng.normal(0, 0.006, n))),        "close": close,        "volume": rng.integers(1e6, 5e6, n).astype(float),    }, index=pd.bdate_range("2023-01-02", periods=n))    return frame, "synthetic"print("=" * 60)try:    from model import Kronos, KronosPredictor, KronosTokenizer    tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")    model = Kronos.from_pretrained("NeoQuasar/Kronos-small")    print("checkpoint + tokenizer loaded")    import torch    device = "cuda:0" if torch.cuda.is_available() else "cpu"    predictor = KronosPredictor(model, tokenizer, device=device, max_context=512)    bars, source = load_bars(TICKER, CONTEXT + PRED_LEN)    history, future = bars.iloc[:-PRED_LEN], bars.iloc[-PRED_LEN:]    print(f"data               {source}, {len(history)} context sessions")    out = predictor.predict(        df=history[["open", "high", "low", "close", "volume"]],        x_timestamp=pd.Series(history.index),        y_timestamp=pd.Series(future.index),        pred_len=PRED_LEN,        T=1.0,        top_p=0.9,        sample_count=SAMPLE_COUNT,        verbose=False,    )    print(f"device             {device}")    print(f"output type        {type(out).__name__}")    print(f"output shape       {getattr(out, 'shape', 'n/a')}")    print(f"output columns     {list(getattr(out, 'columns', []))}")    print(out.head(3) if hasattr(out, "head") else out)    print()    print("NOTE: CRPS for Kronos must come from sampled paths, not a quantile")    print(f"head. sample_count={SAMPLE_COUNT} here; a real run needs far more.")    print("=" * 60)    print(f"PASS - Kronos ran, {PRED_LEN} sessions x {SAMPLE_COUNT} samples")except Exception as exc:    import traceback    traceback.print_exc()    print("=" * 60)    print(f"FAIL - Kronos: {type(exc).__name__}: {exc}")

In [ ]:
# ============================================================# CELL 3 - THE OTHER FOUR FOUNDATION CHECKPOINTS# One load + one call each. Report only - do not debug here.# ============================================================# ** STOP THE SESSION BEFORE RUNNING THIS CELL. **## Not "restart the kernel" - that is not enough. Restarting the kernel on Kaggle# clears Python state but leaves the site-packages tree exactly as a previous# cell's pip left it. Only stopping the session entirely (the power button)# restores the clean image. A kernel restart after an unconstrained install# gives you a fresh interpreter on top of a broken environment, which is the# worst of both: the symptoms move but the cause does not.## Then run cell 1 and this cell only. Cell 2 (Kronos) imports transformers, and# pip cannot replace a module that is already imported.# ============================================================# ROOT CAUSE this cell exists to prevent.## Unconstrained, the installs below drag torch from 2.10.0+cu128 to# 2.14.0+cu130. Kaggle's torchvision 0.25.0 and torchaudio 2.10.0 are compiled# against 2.10.0, so they land partially initialised against the new binary.# transformers imports torch on the way in and breaks with it, which is how# Chronos-2 and TTM came to report##     ImportError: cannot import name 'PreTrainedModel' from 'transformers'## under transformers 4.56.0 - a version where that symbol certainly exists. The# error names transformers; the fault is the torch stack underneath it.## A constraints file pins a version WITHOUT adding a requirement, which is# exactly what is needed: nothing here wants torch, and nothing here may move# it.# ============================================================import sysfrom importlib.metadata import PackageNotFoundErrorfrom importlib.metadata import version as dist_versionSTACK = ("torch", "torchvision", "torchaudio", "transformers")WANTED = {"chronos-forecasting", "timesfm", "uni2ts", "granite-tsfm"}def versions():    out = {}    for name in STACK:        try:            out[name] = dist_version(name)        except PackageNotFoundError:            out[name] = None    return outbefore = versions()print("BEFORE install")for name, value in before.items():    print(f"  {name:<16}{value or 'not installed'}")print()!printf "torch==2.10.0\ntorchvision==0.25.0\ntorchaudio==2.10.0\n" > /tmp/c.txt!echo "--- /tmp/c.txt ---"; cat /tmp/c.txt; echo "------------------"!pip install -q -c /tmp/c.txt chronos-forecasting timesfm uni2ts granite-tsfm!pip install -q -c /tmp/c.txt "transformers<5"after = versions()print()print("AFTER install")for name in STACK:    moved = "" if before[name] == after[name] else f"   <-- MOVED from {before[name]}"    print(f"  {name:<16}{after[name] or 'not installed'}{moved}")print()# --- Guard 1: the torch stack must not have moved -----------------------------moved = [n for n in ("torch", "torchvision", "torchaudio")         if before[n] is not None and before[n] != after[n]]if moved:    print("=" * 60)    print(f"FAIL - the constraint did not hold; {', '.join(moved)} moved.")    print("       Everything below would be testing a broken torch stack.")    print("       Stop the session (power button) and re-run this cell first.")    raise SystemExit("torch stack moved under a constraints file")# --- Guard 2: every package must have installed under the constraint ----------missing = []for name in sorted(WANTED):    try:        dist_version(name)    except PackageNotFoundError:        missing.append(name)if missing:    print("=" * 60)    print(f"FAIL - did not install under the torch constraint: {', '.join(missing)}")    print("       This is a genuine incompatibility, not something to route")    print("       around by letting torch move. It is a go/no-go input:")    print("       either the package supports Kaggle's torch 2.10 stack or it")    print("       needs its own session with its own image.")    print("       Re-run the install without -q to read pip's resolver output.")    raise SystemExit(f"incompatible under constraint: {', '.join(missing)}")# --- The kernel must not be serving a stale transformers ----------------------already_imported = "transformers" in sys.modulesin_memory = getattr(sys.modules.get("transformers"), "__version__", None)if already_imported and str(in_memory) != str(after["transformers"]):    print("=" * 60)    print(f"FAIL - kernel holds transformers {in_memory}, disk has "          f"{after['transformers']}.")    print("       pip cannot replace an imported module. Results below would be")    print("       from the old version. Stop the session and run cell 1 + this")    print("       cell only.")    raise SystemExit("stale transformers in kernel")print("torch stack held, all four packages installed, transformers "      f"{after['transformers']}")print()import warningsimport numpy as npimport pandas as pdwarnings.filterwarnings("ignore")CONTEXT = 512PRED_LEN = 10# Resolved on the second Kaggle run and recorded in configs/models.yaml and# registry.py. The entry point is TimesFM3Forecaster; timesfm.TimesFm was a# guess and does not exist. The observed namespace was:#   ForecastConfig, TimesFM3Forecaster, TimesFM3Torch, TimesFM_2p5_200M_torch,#   configs, timesfm_2p5, timesfm_2p5_torch, torchTIMESFM_ENTRY_POINT = "TimesFM3Forecaster"def load_series(n):    """One univariate close series; these four take a series, not OHLCV."""    try:        import yfinance as yf        frame = yf.download("AAPL", period="3y", progress=False, auto_adjust=True)        if len(frame):            if isinstance(frame.columns, pd.MultiIndex):                frame.columns = frame.columns.get_level_values(0)            column = "Close" if "Close" in frame else "close"            return frame[column].dropna().tail(n).to_numpy(), "yfinance"    except Exception:        pass    rng = np.random.default_rng(20260902)    return 100 * np.exp(np.cumsum(rng.normal(0, 0.015, n))), "synthetic"values, source = load_series(CONTEXT)print(f"context: {len(values)} sessions from {source}\n")results = {}def attempt(name, checkpoint, fn):    try:        outcome = fn()        results[name] = ("LOADED", str(outcome))        print(f"  {name:<14} LOADED   {checkpoint}  -> {outcome}")    except Exception as exc:        detail = f"{type(exc).__name__}: {exc}"        results[name] = ("FAILED", detail)        print(f"  {name:<14} FAILED   {checkpoint}")        print(f"                 {detail[:200]}")def run_chronos():    import torch    from chronos import BaseChronosPipeline    pipe = BaseChronosPipeline.from_pretrained(        "amazon/chronos-2",        device_map="cuda" if torch.cuda.is_available() else "cpu")    out = pipe.predict(torch.tensor(values, dtype=torch.float32), PRED_LEN)    return tuple(np.asarray(out).shape)def run_timesfm():    """Use the entry point the last run identified; still print the namespace.    If a future release moves it again, the print is what makes that visible    immediately instead of after another wasted session.    """    import timesfm    print(f"    timesfm version  {getattr(timesfm, '__version__', 'unknown')}")    public = sorted(n for n in dir(timesfm) if not n.startswith("_"))    print(f"    dir(timesfm)     {public}")    if not hasattr(timesfm, TIMESFM_ENTRY_POINT):        raise AttributeError(            f"timesfm.{TIMESFM_ENTRY_POINT} is gone; namespace printed above. "            "Update TIMESFM_ENTRY_POINT here and in configs/models.yaml.")    cls = getattr(timesfm, TIMESFM_ENTRY_POINT)    print(f"    entry point      timesfm.{TIMESFM_ENTRY_POINT}")    model = cls()    for loader in ("load_checkpoint", "from_pretrained", "compile"):        if hasattr(model, loader):            try:                getattr(model, loader)()                print(f"    called           .{loader}()")            except Exception as exc:                print(f"    .{loader}() raised {type(exc).__name__}: {str(exc)[:90]}")    return f"constructed via timesfm.{TIMESFM_ENTRY_POINT}"def run_moirai():    from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module    module = Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small")    forecast = Moirai2Forecast(module=module, prediction_length=PRED_LEN,                               context_length=len(values),                               target_dim=1, feat_dynamic_real_dim=0,                               past_feat_dynamic_real_dim=0)    predictor = forecast.create_predictor(batch_size=1)    return f"constructed {type(predictor).__name__}"def run_ttm():    import torch    from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction    model = TinyTimeMixerForPrediction.from_pretrained(        "ibm-granite/granite-timeseries-ttm-r2")    ctx = model.config.context_length    x = torch.tensor(values[-ctx:], dtype=torch.float32).reshape(1, ctx, 1)    with torch.no_grad():        out = model(past_values=x)    return tuple(out.prediction_outputs.shape)print("=" * 60)attempt("Chronos-2", "amazon/chronos-2", run_chronos)attempt("TimesFM-2.5", "google/timesfm-2.5-200m-pytorch", run_timesfm)attempt("Moirai-2.0", "Salesforce/moirai-2.0-R-small", run_moirai)attempt("TTM", "ibm-granite/granite-timeseries-ttm-r2", run_ttm)print("=" * 60)final = versions()drifted = [n for n in ("torch", "torchvision", "torchaudio")           if after[n] != final[n]]if drifted:    print(f"WARNING: {', '.join(drifted)} moved during the model loads themselves")loaded = [k for k, (s, _) in results.items() if s == "LOADED"]failed = [k for k, (s, _) in results.items() if s == "FAILED"]print(f"torch {final['torch']} | transformers {final['transformers']}")print(f"loaded ({len(loaded)}): {', '.join(loaded) if loaded else 'none'}")print(f"failed ({len(failed)}): {', '.join(failed) if failed else 'none'}")print("PASS - all four loaded" if not failed else      f"FAIL - {len(failed)} of 4 did not load: {', '.join(failed)}")print()print("Reporting only. Do not debug here - record what broke, then decide")print("whether the integration cost is worth the GPU budget.")

In [ ]:
# ============================================================# CELL 4 - NEURALFORECAST SMOKE TEST (DLinear, MQLoss, 9-point grid)# ============================================================# RUN THE REAL NEURAL TIER IN ITS OWN KAGGLE SESSION.## Lightning forks worker processes, and it cannot fork once CUDA has been# initialised in the interpreter. Cells 2 and 3 both initialise it, so running# this cell after them raised:##     RuntimeError: Lightning can't create new processes if CUDA is already#     initialized## The first attempt at this passed num_workers_loader=0, which failed with##     TypeError: Trainer.__init__() got an unexpected keyword argument#     'num_workers_loader'## because that parameter does not exist in neuralforecast 3.2.1. Anything the# model constructor does not name falls into **trainer_kwargs and is handed# straight to pl.Trainer, which rejects it. Verified against the 3.2.1 source,# not assumed: `grep -rn num_workers_loader neuralforecast/` returns nothing,# and DLinear.__init__ names `dataloader_kwargs=None` instead. The chain is##     DLinear(dataloader_kwargs={"num_workers": 0})#       -> BaseModel.dataloader_kwargs#       -> TimeSeriesDataModule(**dataloader_kwargs)#       -> the DataLoader## while accelerator and devices legitimately travel through **trainer_kwargs to# pl.Trainer. So: dataloader settings in dataloader_kwargs, trainer settings# loose. Zero workers is a workaround, not a fix - it serialises data loading,# which is fine for one epoch on five tickers and wrong for seven models over# 50 tickers and four folds. The foundation tier and the neural tier belong in# separate sessions: a fresh kernel per tier, so the neural run gets real# dataloader workers and an uninitialised CUDA context.# ============================================================import osimport warningsimport numpy as npimport pandas as pdwarnings.filterwarnings("ignore")TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "JNJ"]HORIZON = 1LOOKBACK = 60# The grid must match configs/eval.yaml exactly - training.quantiles and the# evaluation grid are checked for equality at load time by# sp500bench.config.load_configs. If the repo is not on this machine the literal# is used, and the source is printed so a silent divergence is visible.GRID = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]grid_source = "literal fallback (repo not found)"for candidate in ("configs/eval.yaml", "../configs/eval.yaml",                  "/kaggle/working/sp500-forecast-benchmark/configs/eval.yaml"):    if os.path.exists(candidate):        import yaml        GRID = yaml.safe_load(open(candidate))["quantiles"]        grid_source = candidate        breakprint(f"quantile grid ({len(GRID)}): {GRID}")print(f"source: {grid_source}\n")def load_panel(tickers):    try:        import yfinance as yf        frame = yf.download(tickers, period="3y", progress=False, auto_adjust=True)        close = frame["Close"] if "Close" in frame else frame["close"]        long = (close.stack().rename("price").reset_index())        long.columns = ["ds", "unique_id", "price"]        long["y"] = long.groupby("unique_id")["price"].transform(            lambda s: np.log(s).diff())        return long.dropna()[["unique_id", "ds", "y"]], "yfinance"    except Exception:        rng = np.random.default_rng(20260902)        dates = pd.bdate_range("2023-01-02", periods=600)        rows = [pd.DataFrame({"unique_id": t, "ds": dates,                              "y": rng.normal(0, 0.018, len(dates))})                for t in tickers]        return pd.concat(rows, ignore_index=True), "synthetic"print("=" * 60)try:    from neuralforecast import NeuralForecast    from neuralforecast.losses.pytorch import MQLoss    from neuralforecast.models import DLinear    panel, source = load_panel(TICKERS)    print(f"panel              {source}, {len(panel):,} rows, "          f"{panel.unique_id.nunique()} tickers")    import torch    on_gpu = torch.cuda.is_available()    model = DLinear(        h=HORIZON, input_size=LOOKBACK, loss=MQLoss(quantiles=GRID),        max_steps=1, val_check_steps=1, enable_progress_bar=False,        logger=False, scaler_type="standard",        # No forked dataloader workers. In 3.2.1 this is dataloader_kwargs,        # NOT num_workers_loader - see the note at the top of the cell.        dataloader_kwargs={"num_workers": 0},        # Pin to one device explicitly rather than letting Lightning choose a        # strategy; "auto" can select a spawning one on a multi-GPU box.        accelerator="gpu" if on_gpu else "cpu",        devices=1,    )    import neuralforecast    print(f"neuralforecast     {neuralforecast.__version__}")    print(f"trainer            accelerator={'gpu' if on_gpu else 'cpu'}, devices=1")    print('dataloader         dataloader_kwargs={"num_workers": 0}')    nf = NeuralForecast(models=[model], freq="B")    nf.fit(df=panel, verbose=False)    forecast = nf.predict()    columns = list(forecast.columns)    print(f"forecast shape     {forecast.shape}")    print(f"forecast columns   {columns}")    # Do not assume the naming scheme - count what actually came back. Different    # neuralforecast versions spell MQLoss outputs differently (-q-1.0, -lo-98,    # -median), and the evaluation layer has to read whatever this is.    output_columns = [c for c in columns if c not in ("unique_id", "ds")]    print(f"\n{len(output_columns)} output column(s) for {len(GRID)} quantiles:")    for c in output_columns:        print(f"  {c}")    emitted = len(output_columns)    print("=" * 60)    if emitted == len(GRID):        print(f"PASS - DLinear emitted all {len(GRID)} quantile columns")        print("       Reminder: run the real neural tier in a fresh session.")    else:        print(f"FAIL - expected {len(GRID)} quantile columns, got {emitted}")        print("       Map these names to the q0.01..q0.99 convention in")        print("       sp500bench.config.quantile_column before any real run.")except Exception as exc:    import traceback    traceback.print_exc()    print("=" * 60)    print(f"FAIL - neuralforecast: {type(exc).__name__}: {exc}")